# Week 10 EDA and Data Cleaning

# 1.1 Setup and Data Visualization

In [18]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", None)
RANDOM_STATE = 42


local_path = "../data/raw/student_performance_predictions/student_performance_updated_1000.csv"
github_url = "https://raw.githubusercontent.com/fredd-a/Intro-to-AI-Project/main/data/raw/student_performance_predictions/student_performance_updated_1000.csv"

if os.path.exists(local_path):
    df = pd.read_csv(local_path) #for online testing on colab
else:
    df = pd.read_csv(github_url) #for local, after cloning the repo

print(df.shape)
df.head()
df.info()
df.isna().sum()

(1000, 12)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 12 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   StudentID                  960 non-null    float64
 1   Name                       966 non-null    object 
 2   Gender                     952 non-null    object 
 3   AttendanceRate             960 non-null    float64
 4   StudyHoursPerWeek          950 non-null    float64
 5   PreviousGrade              967 non-null    float64
 6   ExtracurricularActivities  957 non-null    float64
 7   ParentalSupport            978 non-null    object 
 8   FinalGrade                 960 non-null    float64
 9   Study Hours                976 non-null    float64
 10  Attendance (%)             959 non-null    float64
 11  Online Classes Taken       975 non-null    object 
dtypes: float64(8), object(4)
memory usage: 93.9+ KB


,0
StudentID,40
Name,34
Gender,48
AttendanceRate,40
StudyHoursPerWeek,50
PreviousGrade,33
ExtracurricularActivities,43
ParentalSupport,22
FinalGrade,40
Study Hours,24


# 1.2 Drop Noisy Columns

In [19]:
#The code below shows "Attendance%" and "StudyHours" to be noisy as indicated by the dataset owner
#df[["AttendanceRate", "Attendance (%)"]].corr()
#df[["StudyHoursPerWeek", "Study Hours"]].corr()

#removing noisy/uneccesary columns
df_clean = df.drop(columns=["Attendance (%)", "Study Hours", "Name"])
print(f"Dropped columns (Attendance (%), Study Hours, Name)")
print("New DF: ",df_clean.shape)



Dropped columns (Attendance (%), Study Hours, Name)
New DF:  (1000, 9)


# 1.3 Drop Missing Rows (NA StudentID)

In [20]:
#cleaning rows with NA student ids, and reformatting
df_clean = df_clean.dropna(subset=["StudentID"])
df_clean["StudentID"] = df_clean["StudentID"].astype(int)
df_clean = df_clean.set_index("StudentID")

# 1.4 Fill-in Categorical Columns with Mode

In [25]:
categorical_columns = ["Gender", "ParentalSupport", "Online Classes Taken"]
for col in categorical_columns:
  missing = df_clean[col].isna().sum()
  mode_value = df_clean[col].mode()[0]
  df_clean[col] = df_clean[col].fillna(mode_value).infer_objects(copy=False)
  #(.infer_objects(copy=False) used after from warning message
  print(f"{col}: filled {missing} missing values with mode {mode_value}")


Gender: filled 0 missing values with mode Male
ParentalSupport: filled 0 missing values with mode High
Online Classes Taken: filled 0 missing values with mode True


# 1.5 Fill-in Numerical Columns with Median

In [22]:
numerical_columns=["AttendanceRate", "StudyHoursPerWeek","PreviousGrade",
                   "ExtracurricularActivities","FinalGrade"]
for col in numerical_columns:
  missing = df_clean[col].isna().sum()
  median_value= df_clean[col].median()
  df_clean[col] = df_clean[col].fillna(median_value)
  print(f"{col}: filled {missing} missing values with median {median_value}")

AttendanceRate: filled 39 missing values with median 88.0
StudyHoursPerWeek: filled 49 missing values with median 17.0
PreviousGrade: filled 32 missing values with median 78.0
ExtracurricularActivities: filled 43 missing values with median 1.0
FinalGrade: filled 38 missing values with median 80.0


# 1.6 DF Check

In [23]:
print(df_clean.isna().sum())
print(f"\nFinal DF shape: {df_clean.shape}")
df_clean.head()

Gender                       0
AttendanceRate               0
StudyHoursPerWeek            0
PreviousGrade                0
ExtracurricularActivities    0
ParentalSupport              0
FinalGrade                   0
Online Classes Taken         0
dtype: int64

Final DF shape: (960, 8)


,Gender,AttendanceRate,StudyHoursPerWeek,PreviousGrade,ExtracurricularActivities,ParentalSupport,FinalGrade,Online Classes Taken
StudentID,,,,,,,,
1,Male,85.0,15.0,78.0,1.0,High,80.0,False
2,Female,90.0,20.0,85.0,2.0,Medium,87.0,True
3,Male,78.0,10.0,65.0,0.0,Low,68.0,False
4,Male,92.0,25.0,90.0,3.0,High,92.0,False
5,Female,88.0,18.0,82.0,2.0,Medium,85.0,True


# 1.7 Exporting Cleaned Data

In [26]:
df_clean.to_csv("student_performance_cleaned.csv")